# exp-037 — count-first decomposition arm (Colab)

Runs **exp-037** end to end on a Colab GPU: the `count_first` condition (the model writes `[N hops]`,
then its steps; the count line is stripped before scoring) against the stored unguided arm of the same
model — Mistral-7B (baseline exp-020, re-parsed by exp-032) and Qwen3.5-9B (baseline exp-027).
Everything else is the baseline's: 4-bit, the exp-014 retrieval artifact, pinned 600, greedy, seed 42.

**Before you run**
1. Runtime → Change runtime type → **L4 GPU** (A100 also works; it is faster but costs more units).
2. Colab secrets (key icon, left bar), notebook access on:
   - `HF_TOKEN` — a Hugging Face token whose account has accepted the Mistral-7B-Instruct-v0.3 licence.
   - `GH_TOKEN` — only if the GitHub repo is private (a token with read access).
3. The branch `feature/37-count-first` must be pushed to GitHub (the notebook clones it).
4. `thesis-backup.tar.gz` sits at `MyDrive/thesis-backup.tar.gz`.

**Time (estimate, not measured on Colab):** on the RTX 3090 the baseline decodes took 42 min (exp-020,
Mistral) and 28 min (exp-027, Qwen). An L4 has roughly a third of the 3090's memory bandwidth, so expect
very roughly 1.5–2.5 h per Mistral run and 1–1.5 h per Qwen run, plus ~15 min setup.
Default plan (3 runs) ≈ 4–6 h on L4. See `experiments/exp-037/README.md`.

Outputs are copied to `MyDrive/thesis-exp-037/` after every step and every 10 minutes during a decode
(the run log and prompt logs stream there; `results.json` is written by the runner only at the end of a
decode, so a runtime that dies mid-decode loses that decode).

In [ ]:
# ---- settings (edit here) ----
REPO_URL = "https://github.com/AhmadiJahid/Thesis---QAv2.git"
BRANCH = "feature/37-count-first"
ARCHIVE_ON_DRIVE = "/content/drive/MyDrive/thesis-backup.tar.gz"
ARCHIVE_SHA256 = "33c6072f0e4e46ab94f40f804463c512f439823b80c3e0e9dec0ee1023717d94"
DRIVE_OUT = "/content/drive/MyDrive/thesis-exp-037"
WORK = "/content/work"                      # clone goes to WORK/Thesis---QAv2, data to WORK/thesis-qav2-data
MIN_VRAM_GIB = 12                           # measured peaks on the 3090: 4.8 GiB (Mistral 4-bit), 7.8 GiB (Qwen 4-bit)

# Which steps to run, in order. The hardware control re-runs the baseline's exact command on this GPU.
RUN = {
    "mistral_count_first": True,     # primary arm
    "mistral_unguided_rerun": True,  # hardware control for exp-020 (recommended)
    "qwen_count_first": True,        # second model
    "qwen_unguided_rerun": False,    # hardware control for exp-027 (optional, +1-1.5 h on L4)
}

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, pathlib
pathlib.Path(DRIVE_OUT).mkdir(parents=True, exist_ok=True)
assert os.path.exists(ARCHIVE_ON_DRIVE), f"archive not found: {ARCHIVE_ON_DRIVE}"
print("Drive OK; outputs ->", DRIVE_OUT)

In [ ]:
# ---- GPU / VRAM check ----
import subprocess
q = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader,nounits"],
                   capture_output=True, text=True)
assert q.returncode == 0, "no GPU: Runtime -> Change runtime type -> L4 GPU"
name, mem_mib, driver = [x.strip() for x in q.stdout.strip().splitlines()[0].split(",")]
print(f"GPU {name}, {int(mem_mib)/1024:.1f} GiB, driver {driver}")
assert int(mem_mib) / 1024 >= MIN_VRAM_GIB, f"need >= {MIN_VRAM_GIB} GiB VRAM"
if "L4" not in name and "A100" not in name:
    print("WARNING: not an L4/A100 - timings in the README estimate do not apply")
GPU_INFO = {"name": name, "memory_total_mib": int(mem_mib), "driver": driver}

In [ ]:
# ---- clone the branch ----
import os, subprocess
from google.colab import userdata
os.makedirs(WORK, exist_ok=True)
REPO = f"{WORK}/Thesis---QAv2"
url = REPO_URL
try:
    tok = userdata.get("GH_TOKEN")
    url = REPO_URL.replace("https://", f"https://x-access-token:{tok}@")
except Exception:
    pass  # public repo, or no secret set
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", url, REPO], check=True)
os.chdir(REPO)
SHA = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=True).stdout.strip()
print("repo at", REPO, "HEAD", SHA)

In [ ]:
# ---- environment: the repo's pinned requirements (torch 2.10.0, transformers 5.5.4, bitsandbytes 0.49.2, ...) ----
!pip install -q -r requirements.txt
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0))
assert torch.cuda.is_available()

In [ ]:
# ---- Hugging Face login (Mistral-7B-Instruct-v0.3 is gated) ----
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
from huggingface_hub import whoami
print("HF user:", whoami()["name"])

In [ ]:
# ---- data: copy the archive locally, then extract only what exp-037 reads ----
# Layout per docs/compute.md "Dataset location": WORK/thesis-qav2-data next to the clone.
import shutil, subprocess, time
local_archive = "/content/thesis-backup.tar.gz"
if not os.path.exists(local_archive):
    t = time.time(); shutil.copy(ARCHIVE_ON_DRIVE, local_archive); print(f"copied archive in {time.time()-t:.0f}s")
subprocess.run([
    "python", "scripts/colab_setup_data.py", "--archive", local_archive, "--sha256", ARCHIVE_SHA256,
    "--parts", "musique",
    "--runs", "exp-014/retrieval/s2_pinned600_imbalanced_rerank_top5.jsonl",
              "exp-032/exp-020/eval", "exp-032/exp-027/eval",
], check=True)

In [ ]:
# ---- step runner: streams the log to Drive, samples VRAM, syncs runs/exp-037 to Drive ----
import json, subprocess, threading, time, shutil, datetime

def sync_to_drive():
    if os.path.exists("runs/exp-037"):
        shutil.copytree("runs/exp-037", f"{DRIVE_OUT}/runs/exp-037", dirs_exist_ok=True)

def run_step(step):
    log_path = f"{DRIVE_OUT}/logs/{step}.log"
    os.makedirs(os.path.dirname(log_path), exist_ok=True)
    peak = {"mib": 0}
    stop = threading.Event()
    def sampler():
        last_sync = time.time()
        while not stop.is_set():
            out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                                 capture_output=True, text=True).stdout.strip()
            if out:
                peak["mib"] = max(peak["mib"], int(out.splitlines()[0]))
            if time.time() - last_sync > 600:
                sync_to_drive(); last_sync = time.time()
            stop.wait(10)
    th = threading.Thread(target=sampler, daemon=True); th.start()
    started = time.time()
    with open(log_path, "a", buffering=1) as log:
        log.write(f"=== {step} start {datetime.datetime.now().isoformat()} sha {SHA} gpu {GPU_INFO}\n")
        p = subprocess.Popen(["bash", "experiments/exp-037/run.sh", step], stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            log.write(line)
            if "Processed" not in line:   # keep the notebook output short
                print(line, end="")
        rc = p.wait()
        wall = time.time() - started
        log.write(f"=== {step} rc={rc} wall_s={wall:.0f} peak_vram_mib_sampled={peak['mib']}\n")
    stop.set(); th.join()
    record = {"step": step, "rc": rc, "wall_seconds": round(wall), "peak_vram_mib_sampled_whole_gpu": peak["mib"],
              "gpu": GPU_INFO, "git_sha": SHA, "finished": datetime.datetime.now().isoformat()}
    os.makedirs("runs/exp-037", exist_ok=True)
    with open("runs/exp-037/colab_steps.jsonl", "a") as f:
        f.write(json.dumps(record) + "\n")
    sync_to_drive()
    print(record)
    assert rc == 0, f"{step} failed (rc={rc}); see {log_path}"

In [ ]:
if RUN["mistral_count_first"]:
    run_step("mistral_count_first")
else:
    print("skipped mistral_count_first")

In [ ]:
if RUN["mistral_unguided_rerun"]:
    run_step("mistral_unguided_rerun")
else:
    print("skipped mistral_unguided_rerun")

In [ ]:
if RUN["qwen_count_first"]:
    run_step("qwen_count_first")
else:
    print("skipped qwen_count_first")

In [ ]:
if RUN["qwen_unguided_rerun"]:
    run_step("qwen_unguided_rerun")
else:
    print("skipped qwen_unguided_rerun")

In [ ]:
# ---- paired compares + Holm (CPU, a few minutes) ----
run_step("compare")
run_step("metrics")
shutil.copy("experiments/exp-037/metrics.json", f"{DRIVE_OUT}/metrics.json")
m = json.load(open("experiments/exp-037/metrics.json"))
for arm, a in m["arms"].items():
    cf = a.get("count_first") or {}
    print(arm, "report card:", {k: round(v, 4) for k, v in a["report_card"].items()})
    if cf:
        print("   count status:", cf["status_counts"], "| count==gold (all rows):",
              cf["overall"]["count_eq_gold_rate_all_rows"], "| count==steps emitted (parsed):",
              cf["overall"]["count_eq_steps_emitted_rate_parsed_rows"])
print("F1 size", m["holm_family_F1_primary_overall"]["size"], "Holm-significant",
      m["holm_family_F1_primary_overall"]["count_significant_holm"])
print("F2 size", m["holm_family_F2_per_hop"]["size"], "Holm-significant",
      m["holm_family_F2_per_hop"]["count_significant_holm"])

In [ ]:
# ---- what to copy back to the laptop ----
sync_to_drive()
print(f"Everything is in {DRIVE_OUT}/ on Google Drive. Copy back to the laptop:")
print(f"  {DRIVE_OUT}/runs/exp-037/        -> C:/Users/suuser/thesis/Thesis---QAv2/runs/exp-037/   (gitignored; whole folder)")
print(f"  {DRIVE_OUT}/metrics.json          -> experiments/exp-037/metrics.json   (committed)")
print(f"  {DRIVE_OUT}/logs/*.log            -> runs/exp-037/logs/                 (run logs, wall clock, VRAM)")
print("Then: copy each arm's decomposer config.json into experiments/exp-037/ as the config snapshot,")
print("write experiments/exp-037/notes.md, and complete the exp-037 row in experiments/log.md (commit", SHA, ").")
for root, _, files in os.walk("runs/exp-037"):
    for f in files:
        if f.endswith((".json", ".jsonl", ".md")) and "prompts_log" not in root:
            print("   ", os.path.join(root, f))